# ResNet-18 on Human Action Recognition — train on Colab, compress locally

Fine-tunes torchvision's ImageNet-pretrained `resnet18` to the 15-class Kaggle
"Human Action Recognition (HAR) Dataset". Architecture and transforms are
copied verbatim from
`resnet18/resnet-18-on-human-action-recognition-dataset.ipynb` so the
state_dict this produces loads into `compresskit.har.build_har_resnet18()`
with no shape mismatch.

Saves only the `state_dict()` at the end — deliberately not a pickled module —
so the file has no dependency on this notebook or Colab's environment.
Download it and hand it to `experiments/import_har_checkpoint.py` in the
compresskit repo.

**Before running:** Runtime -> Change runtime type -> GPU.

**You will need:** a Kaggle API token. Get one at kaggle.com -> Settings ->
API -> "Create New Token" (downloads `kaggle.json`). Cell 2 will prompt you
to upload it.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device != "cuda":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, "
          "then Runtime -> Restart session, before continuing.")

## Get the dataset

Uploads your `kaggle.json`, then downloads and unzips the dataset directly
onto the Colab VM (nothing round-trips through your laptop for this part).

**Note:** as distributed, this dataset is NOT an ImageFolder layout. `train/`
is a flat folder of 12,601 images labelled by `Training_set.csv`
(filename,label); `test/` is a flat folder of 5,410 images with
`Testing_set.csv` giving filenames only — **no labels** (it's a Kaggle
submission target, not a set we have ground truth for). The next cells build
our own labelled train/test split out of the 12,601 labelled images instead
of trusting either of Kaggle's folders.

In [ ]:
from google.colab import files

print("Upload your kaggle.json (kaggle.com -> Settings -> API -> Create New Token):")
uploaded = files.upload()

import os
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip install -q kaggle
!kaggle datasets download -d meetnagadia/human-action-recognition-har-dataset -p /content/har --unzip
!find /content/har -maxdepth 2 -type d

## Architecture

Copied verbatim from `compresskit/har.py`'s `build_har_resnet18` — same
layer names (`model.fc` as a 2-element `Sequential`), so the state_dict
handoff lines up exactly. Do not edit this cell.

The original Kaggle notebook this is based on tried to freeze the backbone
and only fine-tune `layer4` + the head, but never actually called
`requires_grad_(False)` on anything first — so it silently fine-tuned the
whole network. This version does the freeze for real: it trains far faster
per epoch and is the standard transfer-learning setup for a 15-class head on
top of ImageNet features.

In [ ]:
HAR_CLASSES = 15

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.BatchNorm1d(num_ftrs),
    nn.Linear(num_ftrs, HAR_CLASSES),
)
# the new head is always trainable regardless of the loop above
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{trainable:,} / {total:,} parameters trainable")

## Data — same CSV-based split as `compresskit/har.py`

Deliberately duplicated rather than imported (Colab can't `import compresskit`
from your laptop's repo). The split is **deterministic by filename, no RNG**:
sort each class's files, take the last 15% as test. That is the one thing
guaranteed to produce an identical split here and in `har.py` without sharing
any random state between two separate machines — do not change
`TEST_FRACTION` or the sort/slice logic without changing both places.

In [ ]:
import csv
from PIL import Image
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TEST_FRACTION = 0.15

data_root = "/content/har/Human Action Recognition"

data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "test": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}


def read_labels(root):
    with open(f"{root}/Training_set.csv", newline="") as f:
        return {row["filename"]: row["label"] for row in csv.DictReader(f)}


def split_filenames(root, test_frac=TEST_FRACTION):
    labels = read_labels(root)
    by_class = {}
    for filename, label in labels.items():
        by_class.setdefault(label, []).append(filename)

    fit_rows, test_rows = [], []
    for label, filenames in by_class.items():
        filenames = sorted(filenames)
        n_test = max(1, round(len(filenames) * test_frac))
        for filename in filenames[:-n_test]:
            fit_rows.append((filename, label))
        for filename in filenames[-n_test:]:
            test_rows.append((filename, label))
    return fit_rows, test_rows


class HARDataset(Dataset):
    def __init__(self, root, rows, class_to_idx, transform):
        self.image_dir = f"{root}/train"
        self.rows = rows
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        filename, label = self.rows[index]
        image = Image.open(f"{self.image_dir}/{filename}").convert("RGB")
        return self.transform(image), self.class_to_idx[label]


class_names = sorted(set(read_labels(data_root).values()))
class_to_idx = {name: i for i, name in enumerate(class_names)}
fit_rows, test_rows = split_filenames(data_root)

image_datasets = {
    "train": HARDataset(data_root, fit_rows, class_to_idx, data_transforms["train"]),
    "test": HARDataset(data_root, test_rows, class_to_idx, data_transforms["test"]),
}

BATCH_SIZE = 64
dataloaders = {
    "train": DataLoader(image_datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, drop_last=True),
    "test": DataLoader(image_datasets["test"], batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2),
}
dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "test"]}
print(dataset_sizes)
print(class_names)
assert class_names == sorted(class_names), "class order must match compresskit.har.class_names()

## Train

Only `layer4` and the head have gradients, so this converges in far fewer
epochs than a full fine-tune. Keeps the best test-accuracy checkpoint rather
than the last epoch's, same pattern as `colab/train_resnet18.ipynb`.

In [ ]:
EPOCHS = 15
LR = 1e-3

optimizer = torch.optim.SGD(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR, momentum=0.9, weight_decay=5e-4,
)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = seen = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        correct += int((model(images).argmax(1) == targets).sum())
        seen += targets.size(0)
    return correct / seen


best_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    started = time.perf_counter()
    model.train()
    running_loss = correct = seen = 0

    for images, targets in dataloaders["train"]:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        correct += int((logits.argmax(1) == targets).sum())
        seen += targets.size(0)

    val_acc = evaluate(dataloaders["test"])
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"epoch {epoch+1:2d}/{EPOCHS}  loss {running_loss/seen:.3f}  "
          f"train {correct/seen*100:.2f}%  val {val_acc*100:.2f}%  "
          f"{time.perf_counter()-started:.1f}s")

print(f"\nbest val accuracy: {best_acc*100:.2f}%")

## Save and download

Only the `state_dict`, plus the class order — `import_har_checkpoint.py`
needs `class_names` to confirm the label indices match on both ends, since a
silently-transposed class order would still "load" without error and just be
wrong.

In [ ]:
torch.save(
    {
        "state_dict": best_state,
        "arch": "har_resnet18",
        "accuracy": best_acc,
        "num_classes": HAR_CLASSES,
        "class_names": class_names,
    },
    "har_resnet18.pt",
)
print("saved har_resnet18.pt")

files.download("har_resnet18.pt")